# Exercises week 38

**FYS-STK3155/4155, fall 2026 — deadline Friday September 18 at midnight.**

## Adaptive learning rates and stochastic gradient descent

The aim this week is that you extend last week's gradient descent code with
the adaptive methods AdaGrad, RMSProp and Adam, that you understand what each
of them does to the step and why their learning rate is not the learning rate
of plain gradient descent, and that you replace the full gradient by a
minibatch gradient with epochs and a learning-rate schedule. Everything here is
reused directly in parts f) and h) of Project 1. The code developed in the
Tuesday session (`week38tuesday.ipynb`, Cases 1 and 2) is meant to be reused,
and the Monday notebook (`week38.ipynb`) contains worked examples of every
piece.

After completing these exercises you will have

* derived the bias correction of Adam and the first step of the adaptive
  methods, and shown why they are insensitive to the scale of a feature;
* your own implementation of AdaGrad, RMSProp and Adam in the same
  three-function structure as last week's gradient descent, checked against
  the closed-form solutions of OLS and Ridge;
* scanned the learning rate for each method and related the windows you find
  to $2/\lambda_{\max}$ (Section 4.5) and to what $\gamma$ means for an
  adaptive method (Sections 4.8–4.12);
* a stochastic gradient descent loop with minibatches, epochs and the
  schedule of Eq. (4.40), and studied the batch size and the schedule
  (Section 4.7);
* applied all of it to the Runge function at degree 10, the problem nothing
  converged on last week.

Reading: Chapter 4, Sections 4.7–4.13 of the book; Goodfellow et al.,
Sections 8.1–8.5.

## The data

We keep last week's function,

$$
f(x) = 2 - x + 5x^2, \qquad x \in [-2, 2],
$$

with $n = 100$ points and Gaussian noise of standard deviation $0.5$, fitted
with polynomial features of degree 2 (an easy problem, $\kappa \approx 1.3$)
and degree 5 (a hard one, $\kappa \approx 520$). As last week, standardise the
columns of the design matrix, centre $\boldsymbol{y}$ and use no intercept
column; the closed-form OLS and Ridge solutions are your reference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def exercise_data(n=100, degree=2, noise=0.5, seed=2026):
    """Standardised polynomial design matrix (no intercept column) and centred targets."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(-2.0, 2.0, n)
    y = 2.0 - x + 5.0 * x**2 + noise * rng.standard_normal(n)
    X = np.column_stack([x**k for k in range(1, degree + 1)])
    X_norm = (X - X.mean(axis=0)) / X.std(axis=0)
    return X_norm, y - y.mean()

def closed_form(X, y, lam=0.0):
    """(X^T X + n lambda I)^-1 X^T y, the 1/n convention of Eq. (3.95)."""
    n, p = X.shape
    return np.linalg.solve(X.T @ X + n * lam * np.eye(p), X.T @ y)

def cost(theta, X, y, lam=0.0):
    """Eqs. (4.12) and (4.16)."""
    return np.sum((X @ theta - y) ** 2) / len(y) + lam * theta @ theta

def gradient(theta, X, y, lam=0.0):
    """Eqs. (4.13) and (4.17): your analytical gradient from last week (or jax.grad of cost)."""
    n = len(y)
    return (2.0 / n) * X.T @ (X @ theta - y) + 2.0 * lam * theta

def hessian_eigs(X, lam=0.0):
    n = len(X)
    return np.linalg.eigvalsh((2.0 / n) * X.T @ X + 2.0 * lam * np.eye(X.shape[1]))

X2, y2 = exercise_data(degree=2)
X5, y5 = exercise_data(degree=5)
for X in (X2, X5):
    eigs = hessian_eigs(X)
    print(f"degree {X.shape[1]}: kappa = {eigs.max() / eigs.min():.1f}, 2/lambda_max = {2 / eigs.max():.3f}")

## Exercise 1: what the adaptive methods do to the step (analytical)

The three adaptive methods replace the single learning rate $\gamma$ by a
per-coordinate step $\gamma/\sqrt{v_j}$, where $v_j$ estimates the second
moment of the gradient in coordinate $j$ (Section 4.8).

### 1a)

Unroll the RMSProp recursion of Eq. (4.47),
$\boldsymbol{v}_t = \rho\,\boldsymbol{v}_{t-1} + (1-\rho)\boldsymbol{g}_t^2$ with
$\boldsymbol{v}_0 = \boldsymbol{0}$, to obtain Eq. (4.49),
$\boldsymbol{v}_t = (1-\rho)\sum_{j=1}^{t}\rho^{\,t-j}\boldsymbol{g}_j^2$, and show
that the weights sum to $1 - \rho^t$. After how many steps has a gradient's
weight fallen below $1/e$ of its initial weight, for $\rho = 0.9$ and
$\rho = 0.99$? Compare with AdaGrad's sum, Eq. (4.42), whose weights are all
one.

### 1b)

Now Adam. Assume the gradient is stationary with $\mathbb{E}[g^2]$ constant.
Take the expectation of the unrolled second moment of Eq. (4.52) and derive
Eq. (4.53), $\mathbb{E}[v_t] = (1-\beta_2^{\,t})\,\mathbb{E}[g^2]$. Do the same
for the first moment with $\beta_1$. Explain why dividing by $1-\beta_i^t$,
Eq. (4.54), removes the bias exactly, and evaluate the correction factors at
$t = 1$, $t = 10$ and $t = 1000$ for the default $\beta_1 = 0.9$,
$\beta_2 = 0.999$.

### 1c)

Show that the first step of Adam, Eq. (4.55) at $t = 1$, is
$\Delta\theta_j = -\gamma\,\mathrm{sign}(g_{1,j})$ (up to $\epsilon$),
independent of the size of the gradient, and that the same holds for AdaGrad.
What is the first step of RMSProp, which has no bias correction, with
$\rho = 0.99$? What does this tell you about the meaning of $\gamma$ for these
methods, compared with plain gradient descent? Without bias correction, by
what factor would Adam's first step be too long?

### 1d)

Multiply one feature (one column of $\boldsymbol{X}$) by a constant $c$. Show
that the corresponding component of the gradient of the OLS cost scales by
$c$ and that the AdaGrad/RMSProp/Adam step in that coordinate is unchanged,
whereas the plain gradient descent step scales by $c$ (and the stability bound
$2/\lambda_{\max}$ moves). Why does this *not* mean that standardising the
data is unnecessary for the adaptive methods? (Hint: the Tuesday session's
rotated bowl.)

## Exercise 1 solutions
# Exercise 1a: RMSProp Recursion

## Unrolling the Recursion

Starting with $\boldsymbol{v}_t = \rho\,\boldsymbol{v}_{t-1} + (1-\rho)\boldsymbol{g}_t^2$ and $\boldsymbol{v}_0 = \boldsymbol{0}$:

- $t=1$: $\boldsymbol{v}_1 = (1-\rho)\boldsymbol{g}_1^2$
- $t=2$: $\boldsymbol{v}_2 = (1-\rho)[\rho\boldsymbol{g}_1^2 + \boldsymbol{g}_2^2]$
- $t=3$: $\boldsymbol{v}_3 = (1-\rho)[\rho^2\boldsymbol{g}_1^2 + \rho\boldsymbol{g}_2^2 + \boldsymbol{g}_3^2]$

**Closed form:**
$$\boldsymbol{v}_t = (1-\rho)\sum_{j=1}^{t}\rho^{t-j}\boldsymbol{g}_j^2$$

## Weights Sum to $1 - \rho^t$

The sum $\sum_{j=1}^{t}\rho^{t-j}$ is a geometric series:
$$\sum_{j=1}^{t}\rho^{t-j} = \frac{1-\rho^t}{1-\rho}$$

Multiply by $(1-\rho)$:
$$(1-\rho) \cdot \frac{1-\rho^t}{1-\rho} = 1 - \rho^t$$

## When Does a Gradient's Weight Fall Below $1/e$?

We want $(1-\rho)\rho^{t-j} = \frac{1-\rho}{e}$, which gives $\rho^{t-j} = 1/e$.

Taking logarithms: $t - j = \frac{1}{\ln(1/\rho)}$

**For $\rho = 0.9$:**
$$t - j = \frac{1}{\ln(1.111)} = \frac{1}{0.1054} \approx 9.5 \text{ steps}$$

**For $\rho = 0.99$:**
$$t - j = \frac{1}{\ln(1.0101)} = \frac{1}{0.01005} \approx 99.5 \text{ steps}$$

## Comparison with AdaGrad

- **RMSProp**: $\boldsymbol{v}_t = (1-\rho)\sum_{j=1}^{t}\rho^{t-j}\boldsymbol{g}_j^2$ (forgets old gradients)
- **AdaGrad**: $\boldsymbol{r}_t = \sum_{j=1}^{t}\boldsymbol{g}_j^2$ (cumulative, never forgets)

RMSProp: older gradients fade exponentially
AdaGrad: effective learning rate decays like $1/\sqrt{t}$

# Exercise 1b: Adam's Bias Correction

## Unroll Second Moment (Eq. 4.52)

$$\boldsymbol{v}_t = \beta_2\boldsymbol{v}_{t-1} + (1-\beta_2)\boldsymbol{g}_t^2, \quad \boldsymbol{v}_0 = \boldsymbol{0}$$

Unroll: $\boldsymbol{v}_t = (1-\beta_2)\sum_{j=1}^{t}\beta_2^{t-j}\boldsymbol{g}_j^2$

## Take Expectations (Stationary Gradient)

Assume $\mathbb{E}[g_j^2]$ is constant. Sum is geometric:

$$\sum_{j=1}^{t}\beta_2^{t-j} = \frac{1-\beta_2^t}{1-\beta_2}$$

Therefore (Eq. 4.53):

$$\mathbb{E}[\boldsymbol{v}_t] = (1-\beta_2^t)\mathbb{E}[g^2]$$

## First Moment

Similarly:

$$\boldsymbol{m}_t = (1-\beta_1)\sum_{j=1}^{t}\beta_1^{t-j}\boldsymbol{g}_j$$

$$\mathbb{E}[\boldsymbol{m}_t] = (1-\beta_1^t)\mathbb{E}[g]$$

## Bias Correction (Eq. 4.54)

The moments start at zero, so they're biased by $(1-\beta_i^t)$. **Fix:** divide by this factor:

$$\hat{\boldsymbol{m}}_t = \frac{\boldsymbol{m}_t}{1-\beta_1^t}, \quad \hat{\boldsymbol{v}}_t = \frac{\boldsymbol{v}_t}{1-\beta_2^t}$$

**Why it removes bias exactly:**

$$\mathbb{E}[\hat{\boldsymbol{v}}_t] = \frac{\mathbb{E}[\boldsymbol{v}_t]}{1-\beta_2^t} = \frac{(1-\beta_2^t)\mathbb{E}[g^2]}{1-\beta_2^t} = \mathbb{E}[g^2]$$

## Correction Factors

For $\beta_1 = 0.9$ and $\beta_2 = 0.999$:

- At $t=1$: correction is 10 (first moment) and 1000 (second moment)
- At $t=10$: correction is 1.54 (first moment) and 100 (second moment)  
- At $t=1000$: correction is 1.00 (first moment) and 1.58 (second moment)





# Exercise 1c: First Step of Adaptive Methods

## Adam's First Step (Eq. 4.55 at t=1)

With bias correction:
$$\hat{\boldsymbol{m}}_1 = \frac{\boldsymbol{m}_1}{1-\beta_1} = \boldsymbol{g}_1$$

$$\hat{\boldsymbol{v}}_1 = \frac{\boldsymbol{v}_1}{1-\beta_2} = \boldsymbol{g}_1^2$$

Step:
$$\Delta\boldsymbol{\theta}_j = -\gamma \frac{\hat{m}_{1,j}}{\sqrt{\hat{v}_{1,j}+\epsilon}} = -\gamma \frac{g_{1,j}}{\sqrt{g_{1,j}^2+\epsilon}}$$

For small $\epsilon$:
$$\boxed{\Delta\theta_{1,j} \approx -\gamma \, \mathrm{sign}(g_{1,j})}$$

**Independent of gradient magnitude!**


## AdaGrad's First Step

AdaGrad at $t=1$:
$$r_1 = g_1^2$$

Step:
$$\Delta\theta_{1,j} = -\gamma \frac{g_{1,j}}{\sqrt{g_{1,j}^2+\epsilon}} \approx -\gamma \, \mathrm{sign}(g_{1,j})$$

**Same result:** step length is $\gamma$ per coordinate, independent of gradient size.


## RMSProp's First Step (no bias correction, $\rho=0.99$)

RMSProp at $t=1$:
$$v_1 = (1-\rho)g_1^2 = 0.01 \cdot g_1^2$$

Step:
$$\Delta\theta_{1,j} = -\gamma \frac{g_{1,j}}{\sqrt{0.01 \cdot g_{1,j}^2+\epsilon}} \approx -\gamma \frac{g_{1,j}}{0.1|g_{1,j}|}$$

$$\boxed{\Delta\theta_{1,j} \approx -\gamma \cdot 10 \, \mathrm{sign}(g_{1,j})}$$

RMS has a step 10 times bigger. Not good.



## What This Tells Us About $\gamma$

**Plain GD:** $\Delta\theta = -\gamma g$ — step size is **proportional to gradient magnitude**

**Adam/AdaGrad:** $\Delta\theta \approx -\gamma \, \mathrm{sign}(g)$ — step size is **fixed per coordinate**, independent of gradient magnitude

**Meaning:** $\gamma$ has different interpretations:
- Plain GD: learning *rate* (multiplies gradient)
- Adam/AdaGrad: step *length* (fixed per coordinate)

This is why $\gamma$ ranges differ: Adam can use $\gamma=0.001$ while plain GD might need $\gamma=0.5$.


## Adam Without Bias Correction

Without correction at $t=1$:
$$\Delta\theta_{1,j} = -\gamma \frac{(1-\beta_1)g_{1,j}}{\sqrt{(1-\beta_2)g_{1,j}^2+\epsilon}}$$

$$\approx -\gamma \frac{1-\beta_1}{\sqrt{1-\beta_2}} \, \mathrm{sign}(g_{1,j})$$

With $\beta_1=0.9$ and $\beta_2=0.999$:

$$\text{Factor} = \frac{0.1}{\sqrt{0.001}} = \frac{0.1}{0.0316} \approx 3.16$$

**Without bias correction, the first step would be 3.16× too long.**




# Exercise 1d: Feature Scaling and Adaptive Methods

## Gradient Scales by $c$

Scale column $j$ by $c$: $\boldsymbol{x}_j' = c\boldsymbol{x}_j$

Gradient:
$$g_j' = \frac{2}{n}(c\boldsymbol{x}_j)^T(\boldsymbol{X}\boldsymbol{\theta} - \boldsymbol{y}) = c \cdot g_j$$

**Gradient scales by $c$.**

## Plain GD Step Scales by $c$

$$\Delta\theta_j = -\gamma g_j' = -\gamma c g_j = c \cdot \Delta\theta_j$$

**Step is $c$ times larger.** Need to retune $\gamma$ for different scalings.

## Adaptive Methods Step is Unchanged

When gradient scales by $c$, second moment scales by $c^2$:
$$g_j' = c g_j, \quad v_j' = c^2 v_j$$

Step:
$$\Delta\theta_j' = -\gamma \frac{c g_j}{\sqrt{c^2 v_j + \epsilon}} \approx -\gamma \frac{c g_j}{|c|\sqrt{v_j}} = -\gamma \frac{g_j}{\sqrt{v_j}}$$

**The $c$ cancels!** Step is unchanged. Same $\gamma$ works regardless of scaling.

## Why Standardizing is Still Necessary

Adaptive methods normalize each coordinate **independently**. They only see the **diagonal of the Hessian**, not off-diagonal correlations.

**Polynomial features** are highly correlated: $X = [x, x^2, x^3, \ldots]$

The Hessian has large **off-diagonal terms** representing this correlation.

Adaptive methods don't see these correlations. They normalize each parameter separately, but miss the ill-conditioning in **correlated directions**.

**Solution:** Standardization decorrelates features:
$$x'_j = \frac{x_j - \bar{x}_j}{\sigma_j}$$

This reduces off-diagonal Hessian terms which improves conditioning for **both** plain GD and adaptive methods.


Adaptive methods are **scale-invariant**, but **not rotation-invariant** as they miss correlations.

**Without standardization:** Adaptive methods still stall in correlated directions because they only see the diagonal of the Hessian.

## Exercise 2: implementing AdaGrad, RMSProp and Adam

Extend last week's code with one step function that holds all five
optimisers. The template below has the update rules to fill in; keep the
gradient as an argument so that the analytical gradient and `jax.grad` are
interchangeable, as in part e) of Project 1.

In [ ]:
def optimiser_step(method, theta, g, state, t, gamma, beta=0.9, rho=0.99,
                   beta1=0.9, beta2=0.999, eps=1e-8):
    """One update of theta from the gradient g at step t = 1, 2, ...; state carries the running quantities."""
    if method == "plain":                                    # Eq. (4.10)
        return theta - gamma * g, state

    
    if method == "momentum":                                # Eq. (4.28)
        if "v" not in state:
            state["v"] = np.zeros_like(theta)

        state["v"] = beta * state["v"] + g
        return theta - gamma * state["v"], state

    
    if method == "adagrad":                                  # Eqs. (4.42) and (4.45)
        if "r" not in state:
            state["r"] = np.zeros_like(theta)

        state["r"] = state["r"] + g**2
        return theta - gamma * g / np.sqrt(state["r"] + eps), state

    
    if method == "rmsprop":                                  # Eqs. (4.47) and (4.48)
        if "r" not in state:
            state["r"] = np.zeros_like(theta)

        state["r"] = rho * state["r"] + (1 - rho) * g**2
        return theta - gamma * g / np.sqrt(state["r"] + eps), state

    
    if method == "adam":                                     # Eqs. (4.51), (4.52), (4.54), (4.55)
        if "v" not in state:
            state["v"] = np.zeros_like(theta)
        if "m" not in state:
            state["m"] = np.zeros_like(theta)

        state["m"] = beta1 * state["m"] + (1 - beta1) * g
        state["v"] = beta2 * state["v"] + (1 - beta2) * g**2

        m_hat, v_hat = state["m"] / (1 - beta1**t), state["v"] / (1 - beta2**t)

        return theta - gamma * m_hat / (np.sqrt(v_hat) + eps), state

    
    raise ValueError(f"unknown method {method}")

def optimise(grad, theta0, method, gamma, num_iters=1000, tol=1e-8, **kw):
    """Full-gradient loop; returns all iterates and the number of steps taken."""
    theta, state = np.array(theta0, dtype=float), {}
    history = [theta.copy()]
    for t in range(1, num_iters + 1):
        g = grad(theta)
        theta, state = optimiser_step(method, theta, g, state, t, gamma, **kw)
        history.append(theta.copy())
        if np.linalg.norm(g) < tol:
            break
    return np.array(history), t

### 2a)

Run all five methods on the degree-2 data for OLS and for Ridge with
$\lambda = 0.01$ (remember the $n\lambda\boldsymbol{I}$ of the closed form).
Use $\gamma = 0.5$ for plain gradient descent and momentum and try
$\gamma \in \{0.01, 0.1\}$ for the adaptive methods. Do all of them converge to
the closed-form solution? Plot the distance
$\|\boldsymbol{\theta}_k - \hat{\boldsymbol{\theta}}\|$ against the iteration for
each method.

### 2b)

Repeat for the degree-5 data ($\kappa = 520$), with $\gamma = 0.9 \cdot 2/\lambda_{\max}$
for plain descent and momentum. Report the number of iterations to reach
$\|\boldsymbol{\theta}_k - \hat{\boldsymbol{\theta}}\| < 10^{-6}$ for each method,
and identify the method that does not get there at a constant $\gamma$.
Explain why, using your result from exercise 1c).

### 2c)

Check one method against a library implementation: for the degree-2 OLS
problem, run `optax.adam` (or `torch.optim.Adam`, or any other) with the same
$\gamma$, $\beta_1$, $\beta_2$ and $\epsilon$ from the same starting point, and
compare the first ten iterates with your own. They should agree to rounding.
If they do not, the usual suspects are the bias correction and the placement
of $\epsilon$ (inside or outside the square root).

## 2 Solutions

In [ ]:
def exercise_2a(lam_adaptive):
    """Run all five methods on degree-2 data for OLS and Ridge. Save results."""
    
    X2, y2 = exercise_data(degree=2)
    theta_star_ols = closed_form(X2, y2, lam=0.0)
    theta_star_ridge = closed_form(X2, y2, lam=0.01)
    
    methods = ["plain", "momentum", "adagrad", "rmsprop", "adam"]
    gamma_dict = {
        "plain": 0.5,
        "momentum": 0.5,
        "adagrad": lam_adaptive,
        "rmsprop": lam_adaptive,
        "adam": lam_adaptive
    }
    
    results = {}
    
    # OLS
    print("\nOLS (lambda=0):")
    print("-" * 70)
    results["OLS"] = {}
    
    for method in methods:
        gamma = gamma_dict[method]
        
        def grad_ols(theta):
            return gradient(theta, X2, y2, lam=0.0)
        
        history, steps = optimise(grad_ols, np.zeros(2), method, gamma, num_iters=1000)
        distance = np.linalg.norm(history - theta_star_ols, axis=1)
        
        converged = distance[-1] < 1e-6
        status = "converged" if converged else "did not converge"
        
        print(f"{method:10s} (gamma={gamma:.3f}): {steps:4d} steps, {status}")
        
        results["OLS"][method] = {
            "history": history,
            "distance": distance,
            "steps": steps,
            "converged": converged
        }
    


    results["Ridge"] = {}

    print(f"\nRidge\n")
    for method in methods:
        gamma = gamma_dict[method]
        
        def grad_ridge(theta):
            return gradient(theta, X2, y2, lam=0.01)
        
        history, steps = optimise(grad_ridge, np.zeros(2), method, gamma, num_iters=1000)
        distance = np.linalg.norm(history - theta_star_ridge, axis=1)
        
        converged = distance[-1] < 1e-6
        status = "converged" if converged else "did not converge"
        
        print(f"{method:10s} (gamma={gamma:.3f}): {steps:4d} steps, {status}")
        
        results["Ridge"][method] = {
            "history": history,
            "distance": distance,
            "steps": steps,
            "converged": converged
        }
    
    return results


results_2a_lam_01 = exercise_2a(0.1)
results_2a_lam_001 = exercise_2a(0.01)


In [ ]:
import matplotlib.pyplot as plt


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

methods = ["plain", "momentum", "adagrad", "rmsprop", "adam"]

for method in methods:
    distance = results_2a_lam_01["OLS"][method]["distance"]
    axes[0, 0].semilogy(distance, label=method, linewidth=2)

axes[0, 0].set_xlabel("Iteration")
axes[0, 0].set_ylabel("Distance to optimum")
axes[0, 0].set_title("OLS (lambda=0, adaptive gamma=0.1)")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot Ridge with gamma=0.1
for method in methods:
    distance = results_2a_lam_01["Ridge"][method]["distance"]
    axes[0, 1].semilogy(distance, label=method, linewidth=2)

axes[0, 1].set_xlabel("Iteration")
axes[0, 1].set_ylabel("Distance to optimum")
axes[0, 1].set_title("Ridge (lambda=0.01, adaptive gamma=0.1)")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot OLS with gamma=0.01
for method in methods:
    distance = results_2a_lam_001["OLS"][method]["distance"]
    axes[1, 0].semilogy(distance, label=method, linewidth=2)

axes[1, 0].set_xlabel("Iteration")
axes[1, 0].set_ylabel("Distance to optimum")
axes[1, 0].set_title("OLS (lambda=0, adaptive gamma=0.01)")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot Ridge with gamma=0.01
for method in methods:
    distance = results_2a_lam_001["Ridge"][method]["distance"]
    axes[1, 1].semilogy(distance, label=method, linewidth=2)

axes[1, 1].set_xlabel("Iteration")
axes[1, 1].set_ylabel("Distance to optimum")
axes[1, 1].set_title("Ridge (lambda=0.01, adaptive gamma=0.01)")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def exercise_2b():
    """Run all five methods on degree-5 OLS problem."""
    
    X5, y5 = exercise_data(degree=5)
    theta_star = closed_form(X5, y5, lam=0.0)
    
    # Get Hessian eigenvalues
    eigs = hessian_eigs(X5, lam=0.0)
    lambda_max = eigs[-1]
    kappa = eigs[-1] / eigs[0]
    gamma_plain = 0.9 * (2.0 / lambda_max)
    

    print(f"Condition number kappa = {kappa:.1f}")
    print(f"lambda_max = {lambda_max:.6f}")
    print(f"gamma for plain/momentum = {gamma_plain:.6f}\n")
    
    methods = ["plain", "momentum", "adagrad", "rmsprop", "adam"]
    gamma_dict = {
        "plain": gamma_plain,
        "momentum": gamma_plain,
        "adagrad": 0.1,
        "rmsprop": 0.1,
        "adam": 0.01
    }
    
    results = {}
    
    
    for method in methods:
        gamma = gamma_dict[method]
        
        def grad_func(theta):
            return gradient(theta, X5, y5, lam=0.0)
        
        history, steps = optimise(grad_func, np.zeros(5), method, gamma, num_iters=20000)
        distance = np.linalg.norm(history - theta_star, axis=1)
        
        converged = distance[-1] < 1e-6
        status = "converged" if converged else "did not converge"
        
        print(f"{method:10s} (gamma={gamma:.6f}): {steps:5d} steps {status}")
        
        results[method] = {
            "history": history,
            "distance": distance,
            "steps": steps,
            "converged": converged
        }
    
    return results

# Run it
results_2b = exercise_2b()

Plain GD is 10 times slower that momentum GD which shows its advantage, AdaGrad and RMSProp does not converge, this corresponds to the 1c task where the first jump is too big and RMSPropp stalls for big t, The same for AdaGrad which decays like 1/$\sqrt{t}$ and also stalls.

Adam converges but very slow due to it haveing $\gamma = 0.01$ and GD having $\gamma = 0.318$. This shows that learningrate mean diffrent things for diffrent models, and this i cannot be compreard directly.

In [ ]:
import optax

def exercise_2c():
    X2, y2 = exercise_data(degree=2)
    def grad_func(theta): return gradient(theta, X2, y2, lam=0.0)
    
    our_history, _ = optimise(grad_func, np.zeros(2), "adam", gamma=0.01, num_iters=10)
    
    opt = optax.adam(learning_rate=0.01, b1=0.9, b2=0.999, eps=1e-8)
    opt_state = opt.init(np.zeros(2))
    theta = np.zeros(2)
    optax_history = [np.zeros(2)]
    for i in range(10):
        updates, opt_state = opt.update(grad_func(theta), opt_state)
        theta = optax.apply_updates(theta, updates)
        optax_history.append(theta.copy())
    
    optax_history = np.array(optax_history)
    max_diff = np.linalg.norm(our_history - optax_history)
    print(f"Max difference: {max_diff:.2e}")

results_2c = exercise_2c()

The maximum diffrence between the self produced method and optax is $2\cdot 10^{-6}$ which is small, but not at machince precision. This indicating a small diffrence in methods.

## Exercise 3: the learning rate means something different to each method

### 3a)

For the degree-5 OLS problem, scan $\gamma$ over a logarithmic grid from
$10^{-3}$ to $1$ (about fifteen values) for each of the five methods, and
record the number of iterations to $\|\boldsymbol{\theta}_k - \hat{\boldsymbol{\theta}}\| < 10^{-6}$
(or "did not converge" within 20 000 iterations). Plot iterations against
$\gamma$ for all methods in one figure, on logarithmic axes.

### 3b)

Discuss the figure: where does each method's window of usable $\gamma$ start
and end, and how do the edges relate to $2/\lambda_{\max}$ (plain descent),
$2(1+\beta)/\lambda_{\max}$ (momentum), and to the fact that the adaptive
methods take steps of length at most $\gamma$ per coordinate? Which method is
the least sensitive to $\gamma$, and is it also the fastest at its best
$\gamma$?

### 3c)

Book Table 4.1 and Figure 4.7 compare the five methods at three learning
rates on a problem with $\kappa = 12$ and find that the ranking reverses
between $\gamma = 0.01$ and $\gamma = 0.5$. Reproduce this behaviour on your
degree-2 data (where $\kappa = 1.3$): which method is best at each $\gamma$, and
what is the lesson for how optimisers should be compared in Project 1?

## Solutions

In [ ]:
def exercise_3a():
    X5, y5 = exercise_data(degree=5)
    theta_star = closed_form(X5, y5, lam=0.0)
    
    def grad_func(theta):
        return gradient(theta, X5, y5, lam=0.0)
    
    methods = ["plain", "momentum", "adagrad", "rmsprop", "adam"]
    gammas = np.logspace(-3, 0, 14)
    
    results = {}
    
    for method in methods:
        print(f"\n{method}:")
        results[method] = {"gammas": gammas, "steps": []}
        
        for gamma in gammas:
            history, steps = optimise(grad_func, np.zeros(5), method, gamma, num_iters=20000)
            distance = np.linalg.norm(history - theta_star, axis=1)
            
            converged = steps if distance[-1] < 1e-6 else 20000
            results[method]["steps"].append(converged)
            
            status = "converged" if distance[-1] < 1e-6 else "did not converge"
            print(f"  gamma={gamma:.3e}: {converged:5d} steps {status}")
    
    return results

results_3a = exercise_3a()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

methods = ["plain", "momentum", "adagrad", "rmsprop", "adam"]

for method in methods:
    gammas = results_3a[method]["gammas"]
    steps = results_3a[method]["steps"]
    ax.loglog(gammas, steps, marker='o', label=method, linewidth=2)

ax.set_xlabel("Learning rate (gamma)")
ax.set_ylabel("Iterations to convergence")
ax.set_title("Exercise 3a: Learning rate sensitivity (degree-5 OLS)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3b)
RMSProp: fails completely as we found in 1c

plain GD: Has a small domain of convergrence and upperbound is at $2/\lambda_{max}$.

Moemntum GD: has a wide range of convergence, but does not converge for samll $\gamma$s. It has a bigger upperbound than GD due to $2(1 +\beta)/\lambda_{max}$.

AdaGrad: very norrow convergence range, slow, esentually unuseble in this case.

Adam: Always convergeble, only slower than momentum for some ranges. It does not change too much dependant on the learning rate. Adam is a good an robust method.




In [ ]:
def exercise_3c():
    """Full-batch GD on degree-2 data at different learning rates."""
    
    X2, y2 = exercise_data(degree=2)
    theta_star = closed_form(X2, y2, lam=0.0)
    cost_star = np.sum((X2 @ theta_star - y2)**2) / len(y2)
    
    def grad_func(theta):
        return gradient(theta, X2, y2, lam=0.0)
    
    methods = ["plain", "momentum", "adagrad", "rmsprop", "adam"]
    gammas = [0.01, 0.1, 0.5]
    
    results = {}
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for gamma_idx, gamma in enumerate(gammas):
        print(f"\nGamma = {gamma}:")
        print("-" * 70)
        
        results[gamma] = {}
        
        for method in methods:
            history, steps = optimise(grad_func, np.zeros(2), method, gamma, num_iters=100)
            
            cost_values = []
            for theta in history:
                cost = np.sum((X2 @ theta - y2)**2) / len(y2)
                cost_values.append(cost)
            cost_values = np.array(cost_values)
            
            excess_cost = cost_values - cost_star
            
            results[gamma][method] = {
                "excess_cost": excess_cost,
                "final_cost": cost_values[-1]
            }
            
            final_value = cost_values[-1]
            print(f"{method:10s}: final cost = {final_value:.4f}")
            
            axes[gamma_idx].semilogy(excess_cost, label=method, linewidth=2)
        
        axes[gamma_idx].set_xlabel("Epoch")
        axes[gamma_idx].set_ylabel("Excess cost C(θ) - C(θ*)")
        axes[gamma_idx].set_title(f"gamma = {gamma}")
        axes[gamma_idx].legend()
        axes[gamma_idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results

results_3c = exercise_3c()

| Method | γ = 0.01 | γ = 0.1 | γ = 0.5 |
|--------|----------|---------|---------|
| Plain SGD | 1.0853 | 0.3432 | 0.3432 |
| Momentum | 0.3438 | 0.3435 | 0.3439 |
| AdaGrad | 32.0949 | 17.0234 | 0.4661 |
| RMSProp | 15.8011 | 0.3432 | 0.3432 |
| Adam | 23.8587 | 0.3493 | 0.3440 |

$\gamma = 0.01$: Momentum is best

$\gamma = 0.1$: Plain and RMSPropo is best, but alle except AdaGrad is good here.

$\gamma = 0.5$: The same as $\gamma = 0.1$.

Note for project 1, we can not optimize using a single fixed $\gamma$. Full comparison equires tuning $\gamma$ for each model.

## Exercise 4: stochastic gradient descent

Replace the full gradient by a minibatch gradient, Eq. (4.33), inside an
epoch loop. Reshuffle the data every epoch and split the shuffled indices into
minibatches of size $M$; the update of Eq. (4.34) is one call to your
`optimiser_step`, so every optimiser of exercise 2 works with minibatches
unchanged.

In [ ]:
def make_batches(n, batch_size, rng):
    """Shuffle the indices and split them into minibatches."""
    idx = rng.permutation(n)
    return [idx[i:i + batch_size] for i in range(0, n, batch_size)]

def step_length(t, t0, t1):
    """The schedule of Eq. (4.40)."""
    return t0 / (t + t1)

def sgd(X, y, method="plain", n_epochs=50, batch_size=5, gamma=0.1, schedule=None, lam=0.0, seed=2026, **kw):
    """Minibatch SGD, Eq. (4.34), with any optimiser. schedule=(t0, t1) replaces gamma by Eq. (4.40).
    Returns the iterate after every epoch."""
    rng = np.random.default_rng(seed)
    n, p = X.shape
    theta, state, t = np.zeros(p), {}, 0
    history = [theta.copy()]
    for epoch in range(n_epochs):
        for batch in make_batches(n, batch_size, rng):
            t += 1
            g = gradient(theta, X[batch], y[batch], lam=lam)                                  # the gradient of the cost on this minibatch
            if schedule:
                gamma_t = step_length(t, schedule[0], schedule[1])
            else:
                gamma_t = gamma                             # constant, or the schedule
            theta, state = optimiser_step(method, theta, g, state, t, gamma_t, **kw)
        history.append(theta.copy())
    return np.array(history)

### 4a)

At a fixed point $\boldsymbol{\theta} = (0, 3)$ of the degree-2 problem,
compute the gradient on many random minibatches of size $M = 1, 5, 25$ and
compare their mean and their spread (standard deviation over batches) with the
full gradient. Verify numerically that the spread falls as $1/\sqrt{M}$, and say
in one sentence what this means for the trade-off between batch size and cost
per step.

### 4b)

Run plain SGD with $M = 5$ on the degree-2 data for 100 epochs at
$\gamma = 0.1$ and at $\gamma = 0.02$, and plot the distance to the closed-form
solution against the number of *single-point gradient evaluations* (epochs
times $n$) together with full-batch gradient descent at $\gamma = 0.5$. Which
method is ahead early, which is ahead late, and does SGD with a constant
learning rate converge at all? Relate the size of the plateau to $\gamma$.

### 4c)

Now the schedule of Eq. (4.40). With $(t_0, t_1) = (1, 10)$ the initial
learning rate is $0.1$. Add this run to the plot of 4b). Then move to the
degree-5 data and run the same schedule: what happens? Find a pair $(t_0, t_1)$
with the same initial learning rate that works there, and explain your choice
in terms of the Robbins–Monro conditions $\sum_t \gamma_t = \infty$,
$\sum_t \gamma_t^2 < \infty$.

### 4d)

Study the batch size on the degree-5 data at $\gamma = 0.1$: $M \in \{1, 5, 20, 100\}$
for 100 epochs. One of these runs diverges although $\gamma$ is below
$2/\lambda_{\max}$ of the full Hessian. Which Hessian is relevant for a single
minibatch step, and what is its largest eigenvalue for $M = 1$? (For a single
point it is $2\|\boldsymbol{x}_i\|^2$.)

### 4e)

Combine SGD with Adam on the degree-5 data ($M = 5$, $\gamma \in \{0.01, 0.05\}$,
100 epochs) and compare with plain SGD and with momentum. Which combination
comes closest to the closed-form solution, and which one fails? This is the
setting of Project 1, part h).

## Solution

In [ ]:
def exercise_4a():    
    X2, y2 = exercise_data(degree=2)
    theta = np.array([0.0, 3.0])
    
    batch_sizes = [1, 5, 25]
    n_batches = 100
    rng = np.random.default_rng(2026)
    
    full_grad = gradient(theta, X2, y2, lam=0.0)
    

    print(f"\nFull gradient at θ = {theta}: {full_grad}")
    print(f"Full gradient norm: {np.linalg.norm(full_grad):.6f}\n")
    
    for M in batch_sizes:
        grads = []
        for i in range(n_batches):
            batch_idx = rng.choice(len(y2), size=M, replace=False)
            g = gradient(theta, X2[batch_idx], y2[batch_idx], lam=0.0)
            grads.append(g)
        
        grads = np.array(grads)
        mean_grad = np.mean(grads, axis=0)
        std_grad = np.std(grads, axis=0)
        
        print(f"Batch size M = {M}:")
        print(f"  Mean gradient: {mean_grad}")
        print(f"  Std dev: {std_grad}")
        print(f"  Std norm: {np.linalg.norm(std_grad):.6f}")
        print(f"  Expected 1/sqrt(M) scaling: {1.0 / np.sqrt(M):.6f}")
        print()

exercise_4a()

The ratio std(M) / std(M=1) follows the 1/$\sqrt{M}$ law:
- M=5: 5.204/10.143 = 0.513 $\approx$ 1/√5 = 0.447
- M=25: 2.051/10.143 = 0.202 $\approx$ 1/√25 = 0.200
  
The choise of M is a tradeoff between stable convergence, and cheap cost. Large M cost a lot, but small M increeses gradiant noise.

In [ ]:
def run_4b(batch_size, gamma, name):
    """Run SGD/full-batch and return excess cost."""
    X2, y2 = exercise_data(degree=2)
    theta_star = closed_form(X2, y2, lam=0.0)
    cost_star = np.sum((X2 @ theta_star - y2)**2) / len(y2)
    
    hist = sgd(X2, y2, method="plain", n_epochs=100, batch_size=batch_size, gamma=gamma)
    costb = np.array([np.sum((X2 @ theta - y2)**2) / len(y2) for theta in hist])
    excess = costb - cost_star
    
    print(f"{name}: final excess = {excess[-1]:.2e}")
    return excess

# Run three times
excess_sgd_01 = run_4b(batch_size=5, gamma=0.1, name="SGD M=5, γ=0.1")
excess_sgd_002 = run_4b(batch_size=5, gamma=0.02, name="SGD M=5, γ=0.02")
excess_full = run_4b(batch_size=100, gamma=0.5, name="Full-batch, γ=0.5")

In [ ]:
def plot_4b(sgd_01, sgd_002, full):
    fig, ax = plt.subplots(figsize=(10, 6))
    evals = np.arange(101) * 100

    ax.semilogy(evals, excess_sgd_01, label="SGD M=5, γ=0.1", linewidth=2)
    ax.semilogy(evals, excess_sgd_002, label="SGD M=5, γ=0.02", linewidth=2)
    ax.semilogy(evals, excess_full, label="Full-batch, γ=0.5", linewidth=2)

    ax.set_xlabel("Gradient evaluations")
    ax.set_ylabel("Excess cost")
    ax.set_title("Exercise 4b")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig, ax
    
fig, ax = plot_4b(excess_sgd_01, excess_sgd_002, excess_full)
plt.show()

$\gamma = 0.1$ is ahead early and settels at the noise floor above $\gamma = 0.02$. The full batch converges to near machince percisiton. The platu is $\propto \gamma$ as in Eq. (4.39) and thus the higher $\gamma$ has a higher platau.


In [ ]:
X2, y2 = exercise_data(degree=2)
theta_star = closed_form(X2, y2, lam=0.0)
cost_star = np.sum((X2 @ theta_star - y2)**2) / len(y2)
n = len(y2)

# SGD M=5 with schedule (t0=1, t1=10), initial gamma = t0/t1 = 0.1
hist_schedule = sgd(X2, y2, method="plain", n_epochs=100, batch_size=5, schedule=(1.0, 10.0))
cost_schedule = np.array([np.sum((X2 @ theta - y2)**2) / n for theta in hist_schedule])
excess_schedule = cost_schedule - cost_star

fig, ax = plot_4b(excess_sgd_01, excess_sgd_002, excess_full)
evals = np.arange(101) * n
ax.semilogy(evals, excess_schedule, label="SGD M=5, schedule (1, 10)", linewidth=2, linestyle="--")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

X5, y5 = exercise_data(degree=5)
theta_star_5 = closed_form(X5, y5, lam=0.0)
cost_star_5 = np.sum((X5 @ theta_star_5 - y5)**2) / len(y5)
n5 = len(y5)

# schedule (1, 10) 
hist_5_a = sgd(X5, y5, method="plain", n_epochs=100, batch_size=5, schedule=(1.0, 10.0))
cost_5_a = np.array([np.sum((X5 @ theta - y5)**2) / n5 for theta in hist_5_a])
excess_5_a = cost_5_a - cost_star_5
print(f"Schedule (1, 10): final excess = {excess_5_a[-1]:.2e}")

# schedule (1, 100) 
hist_5_b = sgd(X5, y5, method="plain", n_epochs=100, batch_size=5, schedule=(1.0, 100.0))
cost_5_b = np.array([np.sum((X5 @ theta - y5)**2) / n5 for theta in hist_5_b])
excess_5_b = cost_5_b - cost_star_5
print(f"Schedule (1, 100): final excess = {excess_5_b[-1]:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
evals = np.arange(101) * n5

ax.semilogy(evals, excess_5_a, label="Schedule (1, 10)", linewidth=2)
ax.semilogy(evals, excess_5_b, label="Schedule (1, 100)", linewidth=2)

ax.set_xlabel("Gradient evaluations")
ax.set_ylabel("Excess cost")
ax.set_title("Exercise 4c: Degree-5, finding the right schedule")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 4c: Robbins-Monro conditions

The schedule $\gamma_t = \frac{t_0}{t + t_1}$ satisfies the Robbins-Monro conditions:

**First condition:** $\sum_t \gamma_t = \infty$
- $\sum \frac{1}{t + t_1}$ diverges (harmonic series)
- Guarantees you make infinite total progress toward the optimum

**Second condition:** $\sum_t \gamma_t^2 < \infty$
- $\sum \frac{1}{(t + t_1)^2}$ converges (p-series with p=2)
- Guarantees the noise variance shrinks to zero

For degree-2 with schedule $(1, 10)$: converges quickly because the problem is easy ($\kappa=1.3$).

For degree-5 with schedule $(1, 10)$: stalls because $t_1=10$ is too small relative to the problem difficulty ($\kappa \approx 520$). The learning rate decays before the algorithm reaches the optimum.

With schedule $(1, 100)$: slower decay matches the longer timescale needed for convergence on the harder problem, while still satisfying both Robbins-Monro conditions.

**Key lesson:** larger $t_1$ for larger $\kappa$, ill-conditioned problems need longer to converge, so keep the learning rate from decaying too fast.

In [ ]:
X5, y5 = exercise_data(degree=5)
theta_star_5 = closed_form(X5, y5, lam=0.0)
cost_star_5 = np.sum((X5 @ theta_star_5 - y5)**2) / len(y5)
n5 = len(y5)

# Hessian eigenvalues for full data
H_full = (2.0 / n5) * X5.T @ X5
eigs_full = np.linalg.eigvalsh(H_full)
lambda_max_full = eigs_full[-1]
print(f"Full data: lambda_max = {lambda_max_full:.6f}, 2/lambda_max = {2/lambda_max_full:.6f}\n")

# Run with different batch sizes at gamma = 0.1
batch_sizes = [1, 5, 20, 100]
gamma = 0.1

results_4d = {}

for M in batch_sizes:
    print(f"Batch size M = {M}:")
    
    hist = sgd(X5, y5, method="plain", n_epochs=100, batch_size=M, gamma=gamma)
    costd = np.array([np.sum((X5 @ theta - y5)**2) / n5 for theta in hist])
    excess = costd - cost_star_5
    
    results_4d[M] = excess
    
    # Check if diverged
    if np.isnan(excess[-1]) or excess[-1] > 1e10:
        print(f"  DIVERGED")
    else:
        print(f"  Final excess: {excess[-1]:.2e}")
    
    # Compute Hessian for single point
    if M == 1:
        # For single minibatch, largest eigenvalue is ||x_i||^2 scaled
        norms_sq = np.sum(X5**2, axis=1)
        largest_norm_sq = np.max(norms_sq)
        lambda_max_single = 2 * largest_norm_sq / 1  # single point Hessian
        print(f"  Single point: max ||x_i||^2 = {largest_norm_sq:.6f}")
        print(f"  Effective lambda_max ≈ {lambda_max_single:.6f}")
        print(f"  Stability bound 2/lambda_max ≈ {2/lambda_max_single:.6f}")
        print(f"  gamma = {gamma} > {2/lambda_max_single:.6f} → DIVERGES")
    print()

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
evals = np.arange(101) * n5

for M in batch_sizes:
    if M in results_4d:
        ax.semilogy(evals, results_4d[M], label=f"M={M}", linewidth=2)

ax.set_xlabel("Gradient evaluations")
ax.set_ylabel("Excess cost")
ax.set_title("Exercise 4d: Batch size sensitivity (degree-5, γ=0.1)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 4d: Batch size and the Hessian

With $\gamma = 0.1$, the full-batch Hessian has $\lambda_{\max} \approx 5.65$, so the stability bound is $2/\lambda_{\max} \approx 0.354$. Since $\gamma = 0.1 < 0.354$, full-batch and large minibatches ($M=5, 20, 100$) converge.

However, $M=1$ (single points) diverges even though $\gamma < 2/\lambda_{\max}$ for the full problem.

The relevant Hessian for a single minibatch gradient is not the full-batch Hessian. For a single data point $\mathbf{x}_i$, the local quadratic approximation has curvature proportional to $\|\mathbf{x}_i\|^2$. The largest eigenvalue over all single points is $\max_i \|\mathbf{x}_i\|^2$, which can be much larger than $\lambda_{\max}$ of the full design matrix.

For $M=1$: effective $\lambda_{\max} \approx 2 \max_i \|\mathbf{x}_i\|^2$. If this exceeds $1/\gamma$, the method diverges at that point, even though the full-batch bound is satisfied.



In [ ]:
X5, y5 = exercise_data(degree=5)
theta_star_5 = closed_form(X5, y5, lam=0.0)
cost_star_5 = np.sum((X5 @ theta_star_5 - y5)**2) / len(y5)
n5 = len(y5)

# SGD configurations: M=5, 100 epochs
configs = [
    {"name": "Plain SGD, gamma=0.01", "method": "plain", "gamma": 0.01},
    {"name": "Plain SGD, gamma=0.05", "method": "plain", "gamma": 0.05},
    {"name": "Momentum SGD, gamma=0.01", "method": "momentum", "gamma": 0.01},
    {"name": "Momentum SGD, gamma=0.05", "method": "momentum", "gamma": 0.05},
    {"name": "Adam SGD, gamma=0.01", "method": "adam", "gamma": 0.01},
    {"name": "Adam SGD, gamma=0.05", "method": "adam", "gamma": 0.05}
]

results_4e = {}

for config in configs:
    print(f"\n{config['name']}:")
    
    hist = sgd(X5, y5, method=config["method"], n_epochs=100, batch_size=5, gamma=config["gamma"])
    coste = np.array([np.sum((X5 @ theta - y5)**2) / n5 for theta in hist])
    excess = coste - cost_star_5
    
    results_4e[config["name"]] = excess
    
    if np.isnan(excess[-1]) or excess[-1] > 1e10:
        print(f"  DIVERGED")
    else:
        print(f"  Final excess: {excess[-1]:.2e}")

fig, ax = plt.subplots(figsize=(12, 6))
epochs = np.arange(101)

for name, excess in results_4e.items():
    ax.semilogy(epochs, excess, label=name, linewidth=2)

ax.set_ylim([1e-12, 1e10])
ax.set_xlabel("Epoch")
ax.set_ylabel("Excess cost")
ax.set_title("Exercise 4e: SGD with different optimizers (degree-5, M=5)")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 4e: SGD with different optimizers

Plain SGD ($\gamma = 0.01$) converges.

Momentum SGD ($\gamma = 0.05$) diverges massivly.

Adam with $\gamma = 0.01$ converges smoothly.

Adam with $\gamma = 0.05$ converges faster than $\gamma = 0.01$ and setteles on a lower noise floor.

All except momentum does as intented, Adam seems to be a very good method.

## Exercise 5: the Runge function at degree 10 (Project 1, parts f and h)

Last week nothing converged on the Runge function $f(x) = 1/(1+25x^2)$ on
$[-1, 1]$ ($n = 100$, noise $0.1$) with a degree-10 polynomial:
$\kappa \approx 2 \times 10^6$. Use the Tuesday session's `runge_data` (the same
standardisation as above).

### 5a)

Run momentum and Adam with the full gradient for 30 000 iterations and plot
the excess cost $C(\boldsymbol{\theta}_k) - C(\hat{\boldsymbol{\theta}}_{\mathrm{OLS}})$
against the iteration. What accuracy do they reach, and does either of them
resolve the flat directions? Explain, with the rotated bowl of the Tuesday
session in mind, why a diagonal rescaling cannot fix a problem whose
ill-conditioning comes from strongly correlated columns.

### 5b)

Add Ridge with $\lambda \in \{10^{-3}, 10^{-2}\}$: what are the condition
numbers now, and how many iterations do momentum and Adam need to reach an
excess cost of $10^{-8}$? Compare the Ridge coefficients with the OLS ones and
discuss whether Ridge is "cheating" — what problem has it solved, and is that
the problem you wanted solved? (Part i) of Project 1 answers the last question
with cross-validation.)

### 5c)

Take one Newton step from $\boldsymbol{\theta}_0 = \boldsymbol{0}$ with the exact
Hessian $\frac{2}{n}\boldsymbol{X}^T\boldsymbol{X}$ — or, following week 37, with
`jax.hessian` of your cost — and compare with the closed form. What did Newton
use that none of the first-order methods had, and why can we not simply do
this for a neural network?

### 5d)

Finally, minibatches: run SGD with Adam ($M = 10$, a decaying schedule of
your choice, 500 epochs) on the degree-10 problem and compare the excess cost
per gradient evaluation with the full-gradient Adam run of 5a). Give a
critical assessment, in the spirit of part h) of Project 1: what has the noise
bought, and what has it cost?

In [ ]:
# Your code for exercise 5 here
# Solution 5a)

def runge_data(n=100, degree=6, noise=0.1, seed=2026):
    """
    from tuesday week 38 juptier notebook
    Runge function 1/(1+25x^2) on [-1,1], standardised polynomial features, centred y."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1.0, 1.0, n)
    y = 1.0 / (1.0 + 25.0 * x**2) + noise * rng.standard_normal(n)
    X = np.column_stack([x**k for k in range(1, degree + 1)])
    X_norm = (X - X.mean(axis=0)) / X.std(axis=0)
    return X_norm, y - y.mean()

def exercise_5a(method_gamma=(("momentum", 0.9 * 2 / eigs.max()), ("adam", 0.05),), lam=0.0, plot=True):
    """ 
    Run all methods on degree-10 Runge problem. Returns a dictionary of excess costs for each method/gamma pair.
    method_gamma is a list of (method, gamma) pairs. 
    lam is the regularisation parameter. 
    plot=True will plot the excess costs.
    """
    X10, y10 = runge_data(degree=10) 

    theta = closed_form(X10, y10, lam=lam)
    cost_val = cost(theta, X10, y10, lam=lam)

    eigs = hessian_eigs(X10, lam=lam)

    print(f"kappa = {eigs.max() / eigs.min():.3e}")
    print(f"lambda = {lam}")

    results = {}

    #running over method/gamma pairs
    for method, gamma in method_gamma:
        path, steps = optimise(lambda theta: gradient(theta, X10, y10, lam = lam), np.zeros(10), method, gamma, num_iters=30000)
        excess_cost = np.array([cost(theta, X10, y10, lam=lam) - cost_val for theta in path])

        results[method, gamma] = excess_cost

        print(f"\n{method}, gamma={gamma:.3f}: final excess cost = {excess_cost[-1]:.3e}")
        idx = np.where(excess_cost < 1e-8)[0]
        if len(idx) > 0:
            print(f"{method}, gamma={gamma:.3f}: reached 1e-8 after {idx[0]} iterations")
        else:
            print(f"{method}, gamma={gamma:.3f}: did not reach 1e-8")

    # turn plot on/off
    if plot:
        plt.figure(figsize=(8, 5))

        for (method, gamma), excess_cost in results.items():
            plt.semilogy(excess_cost, label=(f"{method}, gamma={gamma:.3f}"))

        plt.xlabel("Iteration")
        plt.ylabel(r"$C(\theta_k) - C(\hat{\theta})$")
        plt.title(f"Degree 10 Runge problem, lambda={lam}")
        plt.legend()
        plt.grid()
        plt.show()
    return results

method_gamma = (("momentum", 0.9 * 2 / eigs.max()), ("adam", 0.05),)
exercise_5a(method_gamma=method_gamma, lam=0.0, plot=True)


## Solution 5a)
The Hessian has a large condition number $\kappa \approx 2\times 10^6$, giving the cost function different curvature in different directions. It makes both methos converge slowly, and the excess cost only reach 2.203e-03 for momentum and 1.384e-03 for adam. Adam preformes slightly better but is ocsilliates. Since we have a degree 10 polynomial, the columns are strongly correlated. A diagonal rescaling can change the scale of each coordinate, but cannot remove correlations between coordinates or rotate into the hessian eigenbasis.

In [ ]:
#solution 5b)

#looping over 5a) for different lambda values
for lam in [1e-3, 1e-2]:
    exercise_5a(lam=lam, plot=True)

## solution 5b)
The condition number $\kappa \approx 2\times 5^3$ with $\lambda = 0.001"$, where momentum needs 1321- and adam 1207 iterations to reach an excess cost of $10^{-8}$. with $\lambda = 0.01"$ the condition number $\kappa \approx 2\times 5^2$ momentum needs 133- and adam 250 iterations to reach an excess cost of $10^{-8}$. Since the ridge coefficient have the extra therm $n\lambda I$ it penalizes large coefficients. For polynomials of higher degrees, Ridge shrinks large coefficients, while OLS does not. This gives ridge a more stablile fit, even dough the coefficients differ from the OLS solution. It solves the fact that large columns blows up, by penalizing them harder.

In [ ]:
# solution 5c)
def exercise_5c(lam=0.0):
    """ 
    Newtons method for first step vs closed form solution. Compare the two.
    """
    X10, y10 = runge_data(degree=10)

    theta0 = np.zeros(10)

    H = (2.0 / len(y10)) * X10.T @ X10 + 2.0 * lam * np.eye(X10.shape[1])
    g = gradient(theta0, X10, y10, lam=lam)

    theta_newton = theta0 - np.linalg.solve(H, g)
    """ 
    closed form
    n, p = X.shape
    return np.linalg.solve(X.T @ X + n * lam * np.eye(p), X.T @ y)
    """    
    theta_ols = closed_form(X10, y10, lam=lam)

    print(f"theta_newton = {theta_newton}")
    print(f"theta_ols = {theta_ols}")
    print(f"Difference = {np.linalg.norm(theta_newton - theta_ols):.3e}")

    return

exercise_5c(lam=0.0)

## Solution 5c)
The difference is small, for one step. Newton used the whole Hessian matrix and the gradient. Gradient decent only uses the gradient. Doing this for a neural network would take to much compute. Since the Hessian is a $p\times p$ matrix and computing $H\Delta \theta = g$ is a $\mathcal{O}(p^3)$ computation

In [ ]:
# Solution 5d)
def exercise_5d(method_gammad=(("adam", 0.005), ("adam", 0.05)), M=1, lam=0.0, plot=True):
    """
    Basically exercise 5a but with different batch sizes.
    using sgd instead of optimize function.
    """
    X10, y10 = runge_data(degree=10)
    theta_star = closed_form(X10, y10, lam=lam)
    cost_star = cost(theta_star, X10, y10, lam=lam)

    n10 = len(y10)

    results_4e = {}

    for method, gamma in method_gammad:
        print(f"\n{method}, gamma={gamma:.3f}:")
        
        hist = sgd(X10, y10, method=method, n_epochs=500, batch_size=M, gamma=gamma)
        cost10 = np.array([np.sum((X10 @ theta - y10)**2) / n10 for theta in hist])
        excess = cost10 - cost_star
        
        results_4e[(method, gamma)] = excess
        
        if np.isnan(excess[-1]) or excess[-1] > 1e10:
            print(f"  DIVERGED")
        else:
            print(f"  Final excess: {excess[-1]:.2e}")

    if plot:
        plt.figure(figsize=(8, 5))
        
        for (method, gamma), excess_cost in results_4e.items():
            plt.semilogy(excess_cost, label=(f"{method}, gamma={gamma:.3f}"))

        plt.xlabel("Gradient evaluations")
        plt.ylabel(r"$C(\theta_k) - C(\hat{\theta})$")
        plt.title(f"Degree 10 Runge problem, lambda={lam}")
        plt.legend()
        plt.grid()
        plt.show()

    return

In [ ]:
method_gammad = (("adam", 0.005), ("adam", 0.05), ("adam", 0.01),)
exercise_5d(method_gammad=method_gammad, M=10, lam=0.0)

In [ ]:
print("Exercise 5a) result")
exercise_5a(method_gamma=method_gammad, lam=0.0, plot=True)

## Solution 5d)
Minibach gives a higher final cost than full-gradient Adam. For the ebst minibach result the exess cost is about $1.1\times10^{-2}$, while the full-gradient in 5a) reaches $1.4\times10^{-3}$. The advantage of minibaches is it is computationally cheaper, with only $M=10$ out of $n=100$ samples used. This makes the gradient noisy, which causes fluctioations. 

For this small data set, the full gradient is still cheap. By watching the runtime in jupiter notebook, and toggling plot=False, the full-gradient use about 10 times the time to run. For bigger dataset, SDG reduces cost per update and is more efficient.

## Insights you should have gained this week — a self-check

Before you hand in, go through the list below without looking at the notes.
Each row is a question we may ask in the Tuesday session or on Project 1; the
right-hand column is the insight it tests, in one sentence.

| Ask yourself | The insight it tests |
|---|---|
| Adam converged for every learning rate from $10^{-3}$ to $1$ and plain gradient descent only in a narrow window. Does $\gamma$ not matter for Adam? | It matters for speed, not for stability: the adaptive step is at most $\gamma$ per coordinate (division by $\sqrt{\hat v}$), so $\gamma$ is a *length in parameter space* and cannot violate Eq. (4.20) — but the best $\gamma$ was still several times faster than the worst. |
| RMSProp at a constant $\gamma$ never reached $10^{-6}$ on a convex quadratic. Why? | Near the minimum $r \approx g^2$ and the step is $\approx \gamma\,\mathrm{sign}(g)$, a fixed length that cannot shrink: the iterate rattles at radius $\sim\gamma$. It needs a decaying $\gamma$ (Eq. 4.40), or the first moment — which is Adam. |
| AdaGrad was slower than plain gradient descent on the degree-6 Runge problem. Is it broken? | No: $\boldsymbol{r}_t$ never forgets, Eq. (4.42), so the effective learning rate decays like $1/\sqrt t$ whether or not the minimum is near. That is the wrong behaviour on a smooth convex problem and the right one for sparse features, which it was designed for. |
| Why does Adam divide its moments by $1 - \beta_i^t$? | Both averages start from zero, so $\mathbb{E}[v_t] = (1-\beta_2^t)\,\mathbb{E}[g^2]$, Eq. (4.53): at $t=1$ the raw second moment is a thousand times too small and the step would be $\sqrt{1000}/\sqrt{10}\approx 3$ times too long. Dividing by the known factor removes the bias exactly; for large $t$ the correction vanishes. |
| What does "adaptive" mean, in one sentence? | Each coordinate gets its own step $\gamma/\sqrt{v_j}$, with $v_j$ the recent mean square gradient as a proxy for the curvature $\lambda_j$: a diagonal, $\mathcal{O}(p)$ imitation of Newton's $\boldsymbol{H}^{-1/2}$ (Section 4.8). |
| Rotating the bowl by $45^\circ$ left plain gradient descent unchanged and made AdaGrad and RMSProp fail. Why? | Eq. (4.19) depends on the eigenvalues only, which a rotation preserves; the adaptive methods rescale *coordinates* and see only the diagonal of $\boldsymbol{H}$. A valley that is not axis-aligned — a correlated polynomial basis, for instance — has its curvature off the diagonal. |
| Every first-order method stalled at an excess cost of $10^{-3}$ on the degree-10 Runge problem, and one Newton step solved it. What did Newton have? | The full Hessian, off-diagonal entries included: $\kappa = 2\times10^6$ is invisible to it, Eq. (4.8). We cannot afford it for a network ($\mathcal{O}(p^3)$); every method from momentum to Adam is a cheaper approximation to it. |
| Book Table 4.1: no optimiser is best at the same $\gamma$ as any other. What does that mean for comparing methods? | A learning rate is not comparable across optimisers — plain descent multiplies the gradient by $\gamma$, the adaptive methods normalise it first. Compare methods at each one's *best* $\gamma$, found by a scan, never at one shared value. |
| Plain SGD with $M=5$ sat at a distance $5\times10^{-2}$ from the minimum for ninety epochs. Has it converged? | No, it has equilibrated: the minibatch gradient is an unbiased estimate with noise $\propto 1/\sqrt M$ (Section 2.5), and at constant $\gamma$ the iterate is a random walk in a cloud of radius $\propto\sqrt\gamma/\sqrt M$. A longer run does not help; a smaller $\gamma$, a larger $M$ or a decaying $\gamma_t$ does. |
| Full-batch gradient descent beat SGD per gradient evaluation on the degree-2 data. Is SGD overrated? | On a small benign problem the exact gradient is cheap and converges geometrically, Eq. (4.26). For $n = 10^6$ one full gradient costs $10^4$ minibatch steps: per unit of computation SGD is far ahead, and it never needs the whole data set in memory — which is why every framework is built on minibatches. |
| The schedule $(t_0,t_1)=(1,10)$ worked at degree 2 and froze at degree 5. Which condition did it break? | Robbins–Monro needs $\sum_t\gamma_t = \infty$; in a finite run $\gamma_t = t_0/(t+t_1)$ with a small $t_1$ decays so fast that the accumulated step cannot cross a $\kappa = 520$ valley. Keep $\gamma_0 = t_0/t_1$ and enlarge both, so the decay starts later. |
| SGD with $M=1$ diverged at a $\gamma$ below $2/\lambda_{\max}$ of the full Hessian. How? | A minibatch step sees the Hessian of the *minibatch* cost; for one point it is $2\boldsymbol{x}_i\boldsymbol{x}_i^T$ with eigenvalue $2\|\boldsymbol{x}_i\|^2$, far above the average. The full-batch bound is a bound on the average, not on every step. |
| When should I stop a stochastic run? | Not on the gradient: monitor the validation cost and stop when it rises while the training cost still falls (early stopping, Section 4.7.1) — a regularisation method as much as a termination rule. |


### Practicalities

Deliver your solutions (a single Jupyter notebook, with the analytical parts
either typeset in Markdown/LaTeX or as a scanned handwritten note) in Canvas
before **Friday September 18 at midnight**. Collaboration is encouraged — list
your collaborators.